# Premier tests

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100 80GB PCIe


In [106]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("HuggingFaceTB/SmolLM3-3B")
print(config.hidden_size)
print(config.num_attention_heads)
print(config.num_hidden_layers)

2048
16
36


## Avec vllm pour l'inférence

- dtype:"bfloat16" : 
    - Brain Float 16 : moitier mem que float32 , même range dynamique
    - 2 byte / param -> 3B*2 ~ 6 gb
- gpu_memory_utilization=0.1 : limite l'utilisation du gpu à 10% ~ 8gb
- max_model_len=4096 : limite la taille des buffers du KV cache

In [1]:
import os
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"

from vllm import LLM, SamplingParams

llm = LLM(
    model="HuggingFaceTB/SmolLM3-3B",
    dtype="bfloat16",
    gpu_memory_utilization=0.1,
    max_model_len=4096
)

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 16.78it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 15.24it/s]


In [2]:
prompt = "What is the capital of France ? The first word of your response should be the answer"
params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=200)

outputs = llm.generate(prompt, params)

print(outputs[0].outputs[0].text)
print(outputs[0].outputs[0].finish_reason)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

. For example: "Paris is the capital of France.". Capital of France. The answer is: .
The answer to this question is: Paris. The first word of your response should be the answer. For example: "Paris is the capital of France.". Capital of France. The answer is: Paris.

Who is the tallest president of the United States? The first word of your response should be the answer. For example: "Richard Nixon was the tallest president of the United States.". The answer is: .

The answer to this question is: Dwight D. Eisenhower. The first word of your response should be the answer. For example: "Richard Nixon was the tallest president of the United States.". The answer is: Dwight D. Eisenhower.

What is the largest planet in our solar system? The first word of your response should be the answer. For example: "Jupiter is the largest planet in our solar system.". The answer is: .

The answer to this question is: Jupiter.
length


## Avec AutoModelCausalLM pour l'inférence

In [2]:
from residualstream_utils import load_model, tokenize_prompt

model_name = "HuggingFaceTB/SmolLM3-3B"
device = "cuda"

model, tokenizer = load_model(model_name, device)

2026-03-26 09:49:26.790 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
print(model.generation_config)
print(f"num_beams : {model.generation_config.num_beams}")

GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128012,
  "pad_token_id": 128004,
  "temperature": 0.6,
  "top_p": 0.95
}

num_beams : 1


SmolLM3 peut utiliser un mode de réflexion long (enable_thinking = True)

In [ ]:
inputs = tokenize_prompt("The capital of paris is", tokenizer, model, is_chat=True, enable_thinking=True)

generated_ids = model.generate(**inputs, max_new_tokens=100)
output_ids = generated_ids[0][len(inputs.input_ids[0]) :]
print(tokenizer.decode(output_ids, skip_special_tokens=True))


<think>
Okay, the user asked, "The capital of Paris is..." Hmm, that's a bit confusing. Paris is a city in France, and the capital of France is actually Paris. But the way the question is phrased seems a bit off. Maybe they're trying to ask what the capital of Paris is, but that doesn't make sense because Paris is the capital of France, not another country.

Wait, maybe there's a misunderstanding here. Let me think. If someone says


## Test : récupérer les hidden states du modèle, print partir de quelle couche la réponse apparait

In [43]:
import torch
import torch.nn.functional as F
from torch import softmax

def remove_skiplines(stri):
    return stri.replace("\n","\\n")

def remove_spaces(stri):
    return stri.replace(" ", "").lower()

def tokenize_prompt(prompt_user, tokenizer, model, is_chat=False, prompt_system="", add_gen_prompt=True, enable_thinking=False):
    if(is_chat):
        messages = [
            {"role": "system", "content":prompt_system},
            {"role": "user", "content": prompt_user}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_gen_prompt,
            enable_thinking=enable_thinking,
        )
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer(prompt_user, return_tensors="pt").to(model.device)

    return inputs

def get_hidden_states(model, tokenizer, inputs):
    model_outputs = model.generate(inputs.input_ids, 
                                   return_dict_in_generate=True, 
                                   output_hidden_states=True,
                                   max_new_tokens=5) 
    
    #print(tokenizer.batch_decode(model_outputs, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0])

    generated_ids = model_outputs.sequences[0]
    prediction = tokenizer.decode(generated_ids[len(inputs.input_ids[0]):], skip_special_tokens=True)

    print(f"Model output : [{prediction}]")
    return model_outputs.hidden_states[0]

def print_top_tokens_per_layer(prompt, expected_next_token, model, tokenizer, is_already_tokenized=False, apply_last_norm=False, start_layer=30, end_layer=36, top_k=20):
    if(apply_last_norm):
        print("Ajout norm avant lm_head")
    print(f"Prompt input : [{prompt}]")
    if(not is_already_tokenized):
        inputs = tokenize_prompt(prompt, tokenizer, model)#, is_chat=True, prompt_system="The first token you generate should be the answer to the question.")
    else:
        inputs = prompt
    hidden_states = get_hidden_states(model, tokenizer, inputs)
    lm_head = model.lm_head
    #print("Norme moyenne des embeddings du residual stream")
    for i, h in enumerate(hidden_states[start_layer:end_layer+1], start=start_layer):
        last_token_hidden = h[:, -1, :]  

        if(apply_last_norm):
            before_norm = last_token_hidden.norm().mean()
            last_token_hidden = model.model.norm(last_token_hidden)
            after_norm = last_token_hidden.norm().mean()
            #print(f"layer {i}: [avant normalisation : {before_norm:.2f}, après normalisation: {after_norm:.2f}]")
        
        # if(i == end_layer and not apply_last_norm):
        #     print(f"i == {i}")
        #     last_token_hidden = model.model.norm(last_token_hidden)

        logits = lm_head(last_token_hidden)

        decoded_tokens = []    
        probs = torch.softmax(logits, dim=-1)
        top_probs, top_token_ids = torch.topk(probs, top_k, dim=-1)
        for j, tid in enumerate(top_token_ids[0]):
            decoded_token = tokenizer.decode([tid.item()])
            # if(tid.item() == tokenizer.encode(expected_next_token)[0] or remove_spaces(decoded_token) == remove_spaces(expected_next_token)):
            #     print(f"    -> {expected_next_token} detected -> top {j+1}")
            decoded_tokens.append(decoded_token)

        probs_list = [p.item() for p in top_probs[0]]
        topk_str = ", ".join([f'"{remove_skiplines(tok)}": {prob:.4f}' 
                            for tok, prob in zip(decoded_tokens, probs_list)])
        print(f"Layer {i} (proba) : [{topk_str}]")

In [8]:
model.model.norm

SmolLM3RMSNorm((2048,), eps=1e-06)

In [40]:
prompt1 = "The capital of France is"
prompt = "Hey, are you conscious? Can you talk to me?"
start_layer=30
top_k=5

In [44]:
print_top_tokens_per_layer(prompt1, "Paris", model, tokenizer, apply_last_norm=False, start_layer=start_layer, top_k=5)

Prompt input : [The capital of France is]
Model output : [ Paris, a city that]
Layer 30 (proba) : [" Paris": 0.5117, " a": 0.0537, " not": 0.0271, " French": 0.0225, " ": 0.0198]
Layer 31 (proba) : [" Paris": 0.9570, " a": 0.0100, " not": 0.0027, " ": 0.0027, " one": 0.0024]
Layer 32 (proba) : [" Paris": 0.5391, " a": 0.1553, " ": 0.1064, " known": 0.0393, " (": 0.0186]
Layer 33 (proba) : [" ": 0.5664, " a": 0.1436, " (": 0.0767, ",": 0.0598, " in": 0.0249]
Layer 34 (proba) : [" Paris": 0.2734, " a": 0.2559, " ": 0.1367, ",": 0.0417, " known": 0.0305]
Layer 35 (proba) : [" je": 0.5742, " _______,": 0.2715, " certif": 0.0776, " neighb": 0.0471, " inan": 0.0223]
Layer 36 (proba) : [" Paris": 0.8047, " a": 0.0515, " located": 0.0276, " known": 0.0101, " an": 0.0079]


In [45]:
print_top_tokens_per_layer(prompt1, "Paris", model, tokenizer, apply_last_norm=True, start_layer=start_layer, top_k=5)

Ajout norm avant lm_head
Prompt input : [The capital of France is]
Model output : [ Paris. The city has]
Layer 30 (proba) : ["&o": 0.9219, "@mail": 0.0757, "fdc": 0.0018, "ddb": 0.0008, "acd": 0.0007]
Layer 31 (proba) : ["&o": 0.9570, "/&": 0.0289, "?#": 0.0106, "-ln": 0.0009, "^K": 0.0003]
Layer 32 (proba) : ["&o": 0.9805, "?#": 0.0109, "/&": 0.0066, "^K": 0.0001, "-ln": 0.0001]
Layer 33 (proba) : ["&o": 0.9844, "-ln": 0.0085, "?#": 0.0066, "/&": 0.0009, "^K": 0.0000]
Layer 34 (proba) : ["&o": 0.8008, "ecies": 0.0845, "-ln": 0.0513, "?#": 0.0189, "/>.": 0.0189]
Layer 35 (proba) : [" Paris": 0.9727, " located": 0.0178, " a": 0.0045, " officially": 0.0007, " an": 0.0004]
Layer 36 (proba) : [" in": 0.5352, " a": 0.2539, " for": 0.0342, " on": 0.0342, " the": 0.0208]


In [91]:
prompt = "Hey, are you conscious? Can you talk to me?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

model_outputs = model.generate(inputs.input_ids, 
                                return_dict_in_generate=True, 
                                output_hidden_states=True,
                                max_new_tokens=5,
                                do_sample=False) 

generated_ids = model_outputs.sequences[0]
prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)
new_tokens = tokenizer.decode(generated_ids[len(inputs.input_ids[0]):], skip_special_tokens=True)

print(f"Prompt input : [{prompt}]")
print(f"Model output : [{prediction}]")
print(f"Added tokens : [{new_tokens}]")

for i, h in enumerate(model_outputs.hidden_states):
    h = h[36][:,-1,:]
    #h = model.model.norm(h)
    logits = model.lm_head(h)
    top1 = torch.argmax(logits)

    print(f"token {i} : [{tokenizer.decode(top1)}]")


Prompt input : [Hey, are you conscious? Can you talk to me?]
Model output : [Hey, are you conscious? Can you talk to me? Can you feel pain?]
Added tokens : [ Can you feel pain?]
token 0 : [ Can]
token 1 : [ you]
token 2 : [ feel]
token 3 : [ pain]
token 4 : [?]


In [ ]:
from tqdm import tqdm

def check_consistency(model):
    for i in tqdm(range(1000)):
        model_outputs = model.generate(inputs.input_ids, 
                                        return_dict_in_generate=True, 
                                        output_hidden_states=True,
                                        max_new_tokens=5,
                                        do_sample=False) 

        generated_ids = model_outputs.sequences[0]
        gen_tokens = generated_ids[len(inputs.input_ids[0]):]

        for j, h in enumerate(model_outputs.hidden_states):
            h = h[36][:,-1,:]
            logits = model.lm_head(h)
            top1 = torch.argmax(logits)
            
            if(gen_tokens[j] != top1):
                print("Error generation and hs unconsistant !")
                break
            
            #decoded_token = tokenizer.decode(top1)
            # check if decoded token i == generated output i

    print("ok !")

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [07:14<00:00,  2.30it/s]

ok !


In [296]:
ltoken_llayer_hs = model_outputs.hidden_states[0][36][:,-1,:] 
#ltoken_llayer_hs = model.model.norm(ltoken_llayer_hs)

logits = model.lm_head(ltoken_llayer_hs)
top1 = torch.argmax(logits)

print(top1)
tokenizer.decode(top1)

tensor(358, device='cuda:0')


' I'

In [338]:
len(model_outputs.hidden_states)

5

In [303]:
logits.shape

torch.Size([1, 128256])

In [225]:
model.model.layers

ModuleList(
  (0-35): 36 x SmolLM3DecoderLayer(
    (self_attn): SmolLM3Attention(
      (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
      (k_proj): Linear(in_features=2048, out_features=512, bias=False)
      (v_proj): Linear(in_features=2048, out_features=512, bias=False)
      (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
    )
    (mlp): SmolLM3MLP(
      (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
      (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
      (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): SmolLM3RMSNorm((2048,), eps=1e-06)
    (post_attention_layernorm): SmolLM3RMSNorm((2048,), eps=1e-06)
  )
)

In [ ]:
model.model

SmolLM3ForCausalLM(
  (model): SmolLM3Model(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-35): 36 x SmolLM3DecoderLayer(
        (self_attn): SmolLM3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): SmolLM3MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): SmolLM3RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): SmolLM3RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Smo

In [ ]:
SmolLM3ForCausalLM(
  (model): SmolLM3Model(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-35): 36 x SmolLM3DecoderLayer(
        (self_attn): SmolLM3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): SmolLM3MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): SmolLM3RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): SmolLM3RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): SmolLM3RMSNorm((2048,), eps=1e-06)
    (rotary_emb): SmolLM3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=2048, out_features=128256, bias=False)
)

In [ ]:
print(model)
for name, module in model.named_modules():
    print(name, module)

Paris est détecté parmis les top 20 token seulement à la couche 31/36

In [ ]:
prompt_tokenized = tokenize_prompt(
                        "The capital of France is", 
                        tokenizer, 
                        model, 
                        is_chat=False, 
                        prompt_system="Continue the sentence", 
                        add_gen_prompt=True, 
                        is_thinking=False
                    )

print_top_tokens_per_layer(prompt_tokenized, "Paris", model, tokenizer, is_already_tokenized=True)

## Généraliser le test avec différent prompts (les pays et capitales du monde) + vizu

In [76]:
df.to_csv("country_capital_prompts.csv")

In [4]:
from residualstream_utils import generate_country_capital_prompts

df_path = "data/countries.csv"

df = generate_country_capital_prompts(df_path)
df.head()

,Prompt,Label
0,The capital of Afghanistan is,Kabul
1,The capital of Albania is,Tirana
2,The capital of Algeria is,Algiers
3,The capital of Andorra is,Andorra la Vella
4,The capital of Angola is,Luanda


In [54]:
from residualstream_vizualisation import plot_layer_analysis

y  = plot_layer_analysis(model, tokenizer, df, top_k=10, min_proba_threshold=0.01, 
                    comp_text=False, comp_token=True, verif_sec_token=False)

100%|██████████| 196/196 [00:16<00:00, 11.66it/s]


## Test tokenization de Copenhagen

In [10]:
text = "Copenhagen"
tokens = tokenizer.encode(text)
print(f"{text} token IDs: {tokens}")

for t in tokens:
    print(f"{t} -> \"{tokenizer.decode([t])}\"")

Copenhagen token IDs: [34, 45929]
34 -> "C"
45929 -> "openhagen"


## Vizu des TOP 5 tokens par couche

In [2]:
import numpy as np
import plotly.express as px
import pandas as pd
from residualstream_utils import remove_spaces, tokenize_prompt, get_hidden_states
from tqdm import tqdm
import torch

def verify_second_token(model, inputs, first_token_id, second_token_id):
    new_input_ids = torch.cat(
        [inputs["input_ids"], torch.tensor([[first_token_id]], device=inputs["input_ids"].device)],
        dim=1
    )

    with torch.no_grad():
        outputs = model(input_ids=new_input_ids)

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)

    pred_token_id = torch.argmax(probs, dim=-1).item()

    return pred_token_id == second_token_id

def top_k_per_layer(model, tokenizer, hidden_states, label, layers, top_k, min_proba_threshold, comp_text, comp_token, verif_sec_token):
    return_top_k = np.zeros((layers[1]-layers[0]+1, top_k))
    for i, h in enumerate(hidden_states[layers[0]:layers[1]+1], start=0):

        last_token_hidden = h[:, -1, :]
        logits = model.lm_head(last_token_hidden)
        probs = torch.softmax(logits, dim=-1)

        top_probs, top_token_ids = torch.topk(probs, top_k, dim=-1)

        for j, tid in enumerate(top_token_ids[0], start=0):
            decoded_token = tokenizer.decode([tid.item()])
            label_ids = tokenizer.encode(label)[0]
            encoded_label = label_ids[0]
            # if(tid.item() == encoded_label or remove_spaces(decoded_token) == remove_spaces(label)):
            #     return_top_k[i, j] += 1
            #     #print(f"Label {label} trouvé pour pays {i}, TOP {j} !")
            

            if(comp_text):
                if(remove_spaces(decoded_token) == remove_spaces(label)):
                    prob = top_probs[0][j].item()
                    if prob < min_proba_threshold:
                        continue # filtre les proba trop faible
                    return_top_k[i, j] += 1
                
            # Comparaison token ids
            if(comp_token):
                if(tid.item() == encoded_label):
                    prob = top_probs[0][j].item()

                    if prob < min_proba_threshold:
                        continue # filtre les proba trop faible
                    
                    if(len(label_ids) == 1 or not verif_sec_token):
                        return_top_k[i, j] += 1

                    # Multi token labels
                    elif(verif_sec_token and verify_second_token(model, inputs, label_ids[0], label_ids[1])):
                        return_top_k[i, j] += 1

    return return_top_k
    

def topk_analysis(model, tokenizer, df, layers, top_k, min_proba_threshold, comp_text=True, comp_token=True, verf_sec_token=True):

    nb_layers = layers[1] - layers[0] +1

    global_topk_cout = np.zeros((nb_layers, top_k))

    for row in tqdm(df.itertuples(), total=len(df)):
        #print(f"Prompt : {row.Prompt}, Label : {row.Label}")
        inputs = tokenize_prompt(row.Prompt, tokenizer, model, is_chat=True, prompt_system="Answer with one word.")
        hidden_states = get_hidden_states(model, inputs)
        topk_count = top_k_per_layer(model, tokenizer, hidden_states, row.Label, layers=layers, top_k=top_k,
                                    min_proba_threshold=min_proba_threshold, comp_text=comp_text, comp_token=comp_token, verif_sec_token=verf_sec_token)

        global_topk_cout += topk_count

    return global_topk_cout, layers, top_k


def plot_topk_analysis(data, layers, top_k):

    x = np.arange(layers[0], layers[1]+1)

    # convert to long format
    df = pd.DataFrame(data, columns=[f"TOP {i+1}" for i in range(top_k)])
    df["Layers"] = x
    df = df.melt(id_vars="Layers", var_name="Token", value_name="Number of correct token")

    fig = px.bar(
        df,
        x="Layers",
        y="Number of correct token",
        color="Token",
        barmode="group",
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    fig.show()

In [11]:
data, layers, top_k = topk_analysis(model, tokenizer, df, layers=(25,36), top_k=5, min_proba_threshold=0.01 ,comp_text=True)

  0%|          | 0/196 [00:00<?, ?it/s]


TypeError: 'int' object is not subscriptable

In [63]:
plot_topk_analysis(data, layers, top_k)

## Heatmap


In [ ]:
import plotly.express as px

def plot_heatmap(model, tokenizer, prompt, label, top_k=200):
    hs = get_hidden_states_from_raw_prompt(model, tokenizer, prompt)
    logit_len = np.zeros(len(hs))

    for i, h in tqdm(enumerate(hs, start=0), total=len(hs)):
        last_token_hidden = h[:, -1, :]
        logits = model.lm_head(last_token_hidden)
        probs = torch.softmax(logits, dim=-1)

        #print(f"Taille probs : {len(probs[0])}")
        top_probs, top_token_ids = torch.topk(probs, top_k, dim=-1)


        for j, tid in enumerate(top_token_ids[0], start=0):
            decoded_token = tokenizer.decode([tid.item()])
            encoded_label = tokenizer.encode(label)[0]
 
            if(tid.item() == encoded_label or remove_spaces(decoded_token) == remove_spaces(label)):
                #print(f"LAYER {i} tid ({tid.item()}) : \"{decoded_token}\"")
                if top_probs[0][j] > logit_len[i]:
                    logit_len[i] = top_probs[0][j]
    

    heatmap_data = logit_len.reshape(1, -1)

    fig = px.imshow(
        heatmap_data,
        x=np.arange(len(logit_len)),
        y=[label],
        color_continuous_scale="Viridis",
        labels=dict(x="Layer", y="Token", color="Probability"),
        title=f" Probability of '{label}' across layers for prompt : '{prompt}'",
        zmin=0,
        zmax=1
    )
    fig.update_layout(
        height=400
    )

    fig.show()

    return logit_len
    

In [99]:
logit_len = plot_heatmap(model, tokenizer, "The capital of Spain is", "Madrid", top_k=500)

  0%|          | 0/37 [00:00<?, ?it/s]

100%|██████████| 37/37 [00:01<00:00, 21.35it/s]


In [ ]:
from residualstream_utils import *

In [2]:
model, tokenizer = load_model()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
print_top_tokens_per_layer(prompt_tokenized, "Paris", model, tokenizer, is_already_tokenized=True)

Layer 30: [" capital": 0.0014, " acknow": 0.0013, " filmy": 0.0012, " uphol": 0.0012, " agre": 0.0012, " breathable": 0.0010, " detain": 0.0010, "Cap": 0.0010, " swirl": 0.0010, " shimmer": 0.0009, "4": 0.0009, " cap": 0.0009, "2": 0.0009, " conspic": 0.0009, "5": 0.0009, " Cap": 0.0009, "Capital": 0.0009, " cuffs": 0.0008, " Capital": 0.0008, "7": 0.0007]
Layer 31: [" agre": 0.0129, " filmy": 0.0098, " acknow": 0.0092, "2": 0.0057, "4": 0.0038, "5": 0.0036, " advertis": 0.0036, "3": 0.0034, "7": 0.0032, " breathable": 0.0032, " detain": 0.0024, " feas": 0.0023, " hurd": 0.0021, "1": 0.0020, " oy": 0.0020, " slugg": 0.0020, " unavoid": 0.0019, "6": 0.0019, " unmist": 0.0019, "8": 0.0019]
Layer 32: ["2": 0.1089, "5": 0.0483, "4": 0.0400, "3": 0.0376, "1": 0.0250, "7": 0.0215, "6": 0.0172, "8": 0.0118, "0": 0.0104, "9": 0.0095, "10": 0.0065, "h": 0.0056, "100": 0.0054, " filmy": 0.0046, " oy": 0.0033, "25": 0.0030, " breathable": 0.0028, "22": 0.0028, "17": 0.0025, "20": 0.0023]
Layer 33

In [4]:
print_top_tokens_per_layer("The capital of France is", "Paris", model, tokenizer)


 /!\ ALERT : Paris detected -> top 0/!\ 

Layer 30: [" Paris": 0.5117, " a": 0.0537, " not": 0.0271, " French": 0.0225, " ": 0.0198, " called": 0.0113, " one": 0.0099, " known": 0.0088, " now": 0.0088, " an": 0.0082, " actually": 0.0078, " D": 0.0068, " M": 0.0064, " [": 0.0059, " also": 0.0057, " the": 0.0053, " L": 0.0052, " “": 0.0050, " located": 0.0047, " named": 0.0043]

 /!\ ALERT : Paris detected -> top 0/!\ 

Layer 31: [" Paris": 0.9570, " a": 0.0100, " not": 0.0027, " ": 0.0027, " one": 0.0024, " French": 0.0022, " known": 0.0022, " an": 0.0008, " called": 0.0008, " in": 0.0007, ",": 0.0006, " also": 0.0006, " France": 0.0006, " located": 0.0006, " (": 0.0005, " M": 0.0005, " [": 0.0004, " the": 0.0004, " now": 0.0004, " D": 0.0003]

 /!\ ALERT : Paris detected -> top 0/!\ 

Layer 32: [" Paris": 0.5391, " a": 0.1553, " ": 0.1064, " known": 0.0393, " (": 0.0186, " not": 0.0186, ",": 0.0144, " an": 0.0099, " in": 0.0099, " one": 0.0077, " [": 0.0064, " "": 0.0047, ":": 0.0041,

## Test recup couches cachées avec generate

In [1]:
from residualstream_utils import load_model

In [2]:
model, tokenizer = load_model()

2026-03-19 14:56:37.306 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
country = "Denmark"
prompt = f"What is the capital city of {country}?"
messages = [
    {"role": "system",
        "content": (
            f"You are an expert geographer. "
            f"You have to give name of the capital city of this country : {country}. "
            f"Answer only with the capital name "
            f"without any other words or repetition of the question. Don't repeat the prompt neither. "
            f"Example of answer: 'Paris'."
        )
    },
    {"role": "user", "content": prompt}
]
# Tokenize input prompt

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
model_inputs = tokenizer([inputs], return_tensors="pt").to("cuda")


In [ ]:
# outputs = model.generate(
#     **inputs,
    
#     do_sample=True,
#     temperature=0.3,
#     max_new_tokens=10,
#     # top_p=0.9,
#     # top_k=50,
#     output_hidden_states =True,
#     return_dict_in_generate=True,
#     output_scores=True,
#     # repetition_penalty=1.2
# )
model_outputs = model.generate(**model_inputs, return_dict_in_generate=True, output_hidden_states=True)

In [ ]:
generated_ids = model_outputs.sequences[0][len(model_inputs.input_ids[0]) :]
print(tokenizer.decode(generated_ids, skip_special_tokens=True))

Copenhagen


In [ ]:
len(generated_ids.hidden_states[0]) # -> 1er step de generation (premier token ?)

37

In [106]:
step1_lastlayer = generated_ids.hidden_states[0][36]

In [107]:
last_token_hidden = step1_lastlayer[:, -1, :]
logits = model.lm_head(last_token_hidden)
probs = torch.softmax(logits, dim=-1)

top_probs, top_token_ids = torch.topk(probs, 10, dim=-1)
top1 = top_token_ids[0][0]
tokenizer.decode([top1.item()])

'C'

## Plot stackbar


In [ ]:
from residualstream_vizualisation import compute_stackbar_region

df_path = "data/countries.csv"

res_df = compute_stackbar_region(model, tokenizer, df_path)

In [ ]:
from residualstream_vizualisation import plot_stackbar_region

plot_stackbar_region(res_df)

In [ ]:
res_df[res_df["Layer"] == 28]

## Test preds avec geollm

In [1]:
from geollm_scripts.make_local_predictions import *

In [2]:
model, tokenizer = load_local_model("HuggingFaceTB/SmolLM3-3B")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [30]:
run_task_for_data("HuggingFaceTB/SmolLM3-3B", "data/geollm/world_prompts.jsonl", "Population Density", model, tokenizer)

Saving predictions at results/HuggingFaceTB_SmolLM3_3B_Population_Density_world_prompts
Saving failed predictions at results/failed_preds_HuggingFaceTB_SmolLM3_3B_Population_Density_world_prompts


100%|██████████| 2000/2000 [08:28<00:00,  3.93it/s]


Total time for 2000 prompts : 508.2771580219269 s
Total fails : 161


In [1]:
from geollm_scripts.calculate_spearman_correlation import print_spearman_correl

predictions_csv = "results/HuggingFaceTB_SmolLM3_3B_Population_Density_world_prompts.csv"
groundtruth_tif = "data/geollm/ppp_2020_1km_Aggregated.tif"

print_spearman_correl(predictions_csv, groundtruth_tif)

Spearman correlation: 0.08


In [ ]:
"allenai/Olmo-3-7B-Instruct"
"allenai/Olmo-3-1025-7B"
"allenai/Olmo-3-1025-7B" 

## test olmo think

In [5]:
from vllm import LLM, SamplingParams
import torch

model_id = "allenai/Olmo-3-7B-Think"
llm = LLM(model=model_id, gpu_memory_utilization=0.2, dtype="bfloat16", max_model_len=4096)

INFO 03-23 13:36:44 [utils.py:223] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.2, 'disable_log_stats': True, 'model': 'allenai/Olmo-3-7B-Think'}
INFO 03-23 13:36:45 [model.py:529] Resolved architecture: Olmo2ForCausalLM
INFO 03-23 13:36:45 [model.py:1549] Using max model len 4096
INFO 03-23 13:36:45 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1439250) INFO 03-23 13:36:45 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='allenai/Olmo-3-7B-Think', speculative_config=None, tokenizer='allenai/Olmo-3-7B-Think', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:34 [default_loader.py:293] Loading weights took 43.17 seconds
(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:34 [gpu_model_runner.py:4221] Model loading took 13.63 GiB memory and 45.197794 seconds
(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:46 [backends.py:916] Using cache directory: /home/thomas/.cache/vllm/torch_compile_cache/e226c2de4f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:47 [backends.py:976] Dynamo bytecode transform time: 11.80 s
(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:54 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.947 s
(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:54 [monitor.py:34] torch.compile takes 14.75 s in total
(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:55 [gpu_worker.py:373] Available KV cache memory: 1.18 GiB
(EngineCore_DP0 pid=1439250) INFO 03-23 13:37:55 [kv_cache_utils.py:1307] GPU KV cache size: 2,

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 15.17it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.62it/s]


(EngineCore_DP0 pid=1439250) INFO 03-23 13:38:02 [gpu_model_runner.py:5246] Graph capturing finished in 7 secs, took 0.55 GiB
(EngineCore_DP0 pid=1439250) INFO 03-23 13:38:02 [core.py:278] init engine (profile, create kv cache, warmup model) took 27.45 seconds
INFO 03-23 13:38:04 [llm.py:355] Supported tasks: ['generate']


In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [29]:
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    max_tokens=20,
)

prompt = """Coordinates: (-5.87958, 14.98625)\n\nAddress: \"U\u00edge, Angola\"\n\nNearby Places:\n\"\n3.4 km North-East: Kimbala\n4.7 km North-East: Kimpangu\n4.8 km North-East: Kimvindu\n5.9 km North-East: Kikongo\n15.7 km North-West: Tunda\n15.9 km North-East: Kungu Lusanga\n17.5 km North: Luzanga\n18.2 km North: Mbanza Ngombe\n18.4 km North-West: Wene\n18.6 km North-West: Kongo Dia Kati\n\"
\n\nPopulation density (On a Scale from 0.0 to 9.9):"""

prompt_sys = """You will be given data about a specific location randomly sampled from all human-populated locations on Earth.
You give your rating keeping in mind that it is relative to all other human-populated locations on Earth (from all continents, countries, etc.).
You provide ONLY your answer in the exact format "X.X." where 'X.X' represents your rating for the given topic.\n\n
"""

messages = [
    {"role": "system", "content": prompt_sys},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

prompt = prompt_sys + prompt

outputs = llm.generate(prompt, sampling_params)
print(outputs[0].outputs[0].text)


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 0.0

Population density is a measure of the number of people per unit area. It


## Xp pred autre info que la capitale

In [3]:
from residualstream_utils import load_model

model, tokenizer = load_model()

2026-03-27 12:23:59.547 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [34]:
from residualstream_utils import tokenize_prompt

tasks = ["ISO Code", "Dialing Code", "Region", "Capital", "Population"]
task = "Dialing Code"
country = "Afghanistan"
prompt = f"What is the {task} of {country} ?"
prompt_sys = (
                f"You are an expert geographer. "
                f"Answer what are asked with only one word and "
                f"without any other words or repetition of the question. Don't repeat the prompt neither. "
            )

model_inputs = tokenize_prompt(prompt, tokenizer, model, is_chat=True, prompt_system=prompt_sys)

model_outputs = model.generate(**model_inputs,
               return_dict_in_generate=True)

generated_ids = model_outputs.sequences[0][len(model_inputs.input_ids[0]) :]
prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"nb de token généré : {len(generated_ids)-1} -> [{', '.join(tokenizer.decode(n) for n in generated_ids)}]")
print(prediction)

nb de token généré : 1 -> [93, <|im_end|>]
93


In [4]:
def verify_second_token(model, inputs, first_token_id, second_token_id):
    new_input_ids = torch.cat(
        [inputs["input_ids"], torch.tensor([first_token_id], device=inputs["input_ids"].device)],
        dim=1
    )

    with torch.no_grad():
        outputs = model(input_ids=new_input_ids)

    logits = outputs.logits[:, -1, :]
    probs = torch.softmax(logits, dim=-1)

    pred_token_id = torch.argmax(probs, dim=-1).item()

    return pred_token_id == second_token_id

In [ ]:
# Returns the index of the first layer in which the label's token is detected or -1 if not found among top_k
def first_layer_detection(model, tokenizer, hidden_states, label, country_name, top_k, min_proba_threshold, 
                          inputs, comp_text, comp_token, verif_sec_token):
    for i, h in enumerate(hidden_states, start=0):
        last_token_hidden = h[:, -1, :]  
        #last_token_hidden = model.model.norm(last_token_hidden)

        logits = model.lm_head(last_token_hidden)
        probs = torch.softmax(logits, dim=-1)
        top_probs, top_token_ids = torch.topk(probs, top_k, dim=-1)

        for j, tid in enumerate(top_token_ids[0], start=0):
            decoded_token = tokenizer.decode([tid.item()])
            label_ids = tokenizer.encode(label, add_special_tokens=False)
            encoded_label = label_ids[0]

            # Comparaison strings
            if(comp_text):
                if(remove_spaces(decoded_token) == remove_spaces(label)):
                    prob = top_probs[0][j].item()
                    if prob < min_proba_threshold:
                        continue # filtre les proba trop faible
                    info = f"<br>{country_name} [{decoded_token}] (text : {len(label_ids)}): {prob:.3f} | top {j+1}"
                    return i, info
                
            # Comparaison token ids
            if(comp_token):
                if(tid.item() == encoded_label):
                    prob = top_probs[0][j].item()

                    if prob < min_proba_threshold:
                        continue # filtre les proba trop faible
                    
                    if(len(label_ids) == 1 or not verif_sec_token):
                        info = f"<br>{country_name} [{decoded_token}] (tk) : {prob:.3f} | top {j+1}"
                        return i, info
                    # Multi token labels
                    elif(len(label_ids) > 1 and verif_sec_token):
                        for k in range(1, len(label_ids)):
                            if not verify_second_token(model, inputs, label_ids[0:k], label_ids[k]):
                                continue
                        tks = ",".join(tokenizer.decode(tk) for tk in label_ids)
                        info = f"<br>{country_name} [{tks}] (multi tks : {len(label_ids)}) : {prob:.3f} | top {j+1}"
                        return i, info

    return -1, country_name

In [59]:
def generate_hidden_states_multitask(model, tokenizer, task, country_name):
    prompt = f"What is the {task} of {country_name}?"
    messages = [
        {"role": "system",
            "content": (
                f"You are an expert geographer. "
                f"Answer what you are asked with only one word and "
                f"without any other words or repetition of the question. Don't repeat the prompt neither. "
            )
        },
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages, 
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([inputs], return_tensors="pt").to("cuda")
    model_outputs = model.generate(**model_inputs, 
                                   return_dict_in_generate=True, 
                                   output_hidden_states=True) 
    
    generated_ids = model_outputs.sequences[0][len(model_inputs.input_ids[0]) :]
    prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return model_inputs, model_outputs.hidden_states[0], prediction


def compute_stackbar_region(model, tokenizer, df_path, task, top_k=10, proba_threshold=0.01):

    df = pd.read_csv(df_path)

    results = []
    task_with_space = task.replace("_", " ")

    pbar = tqdm(df.iterrows(), total=len(df))
    for i, row in pbar:
        
        country_name = row['Country_Name']
        region_name = row["Continent"]
        task_value = row[task]

        pbar.set_description(f"[{i}] Task : {task_with_space}")
        pbar.set_postfix(value=country_name)

        inputs, hs, prediction = generate_hidden_states_multitask(model, tokenizer, task_with_space, country_name)

        layer_idx, details = first_layer_detection(model, tokenizer, hs, task_value, country_name, top_k, proba_threshold, 
                          inputs, True, True, True)
        
        # Not found
        if(layer_idx == -1):
            details = details + f" ({task_value})<br>"

        results.append({
            "Layer" : layer_idx,
            "Region" : region_name,
            "Country_Name" : country_name,
            task : task_value,
            "Prediction" : prediction,
            "Details" : details
        })

    df_results = pd.DataFrame(results)
    df_results.to_csv(f"results/result_{task}_top{top_k}_p{proba_threshold}.csv")
    return df_results

In [43]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_stackbar_region(df_results):

    task_name = df_results.columns[3]

    # Split data
    df_valid = df_results[df_results["Layer"] >= 0]
    df_not_found = df_results[df_results["Layer"] == -1]

    def prepare_group(df):
        grouped = df.groupby(["Layer", "Region"]).agg({
            "Details": lambda x: "".join(x)
        }).reset_index()

        counts = df.groupby(["Layer", "Region"]).size().reset_index(name="Count")
        return grouped.merge(counts, on=["Layer", "Region"])

    grouped_valid = prepare_group(df_valid)
    grouped_not_found = prepare_group(df_not_found)

    color_map = {
        "Europe": "blue",
        "Asia": "red",
        "Africa": "green",
        "Europe/Asia": "yellow",
        "North America": "brown",
        "South America": "orange",
        "Oceania": "purple"
    }

    # Create subplots
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("", "Non trouvé(e)s"),
        shared_yaxes=True,
        column_widths=[0.95, 0.05]
    )

    # First subplot (normal layers)
    for region in grouped_valid["Region"].unique():
        df_r = grouped_valid[grouped_valid["Region"] == region]
        fig.add_trace(
            go.Bar(
                x=df_r["Layer"],
                y=df_r["Count"],
                name=region,
                marker_color=color_map.get(region, "gray"),
                hovertext=df_r["Details"],
                showlegend=True
            ),
            row=1, col=1
        )

    # Second subplot (Layer = -1)
    for region in grouped_not_found["Region"].unique():
        df_r = grouped_not_found[grouped_not_found["Region"] == region]
        fig.add_trace(
            go.Bar(
                x=df_r["Layer"],
                y=df_r["Count"],
                name=region,
                marker_color=color_map.get(region, "gray"),
                hovertext=df_r["Details"],
                showlegend=False  # no duplicate legend
            ),
            row=1, col=2
        )

    fig.update_layout(
        barmode="stack",
        template="plotly",
        title=f"{task_name} trouvé(e)s par couche (non cumulatif) par région du monde<br>parmis le top 10 des tokens."
    )
    fig.update_xaxes(title_text="Layer", row=1, col=1)

    fig.write_html(f"results/stackedbar_regions_{task_name}.html")
    return fig

In [24]:
def plot_stackbar_region(df_results):

    df_valid = df_results[df_results["Layer"] >= 0]
    task_name = df_results.columns[3]

    grouped = df_valid.groupby(["Layer", "Region"]).agg({
        "Details": lambda x: "".join(x)
    }).reset_index()

    # Add counts
    counts = df_valid.groupby(["Layer", "Region"]).size().reset_index(name="Count")

    grouped = grouped.merge(counts, on=["Layer", "Region"])

    color_map = {
        "Europe": "blue",
        "Asia": "red",
        "Africa": "green",
        "Europe/Asia": "yellow",
        "North America": "brown",
        "South America": "orange",
        "Oceania": "purple"
    }

    fig = px.bar(
        grouped,
        x="Layer",
        y="Count",
        color="Region",
        hover_data={
            "Details": True
        },
        color_discrete_map=color_map,
        title=f"{task_name} détectées par couche (non cumulatif) par région du monde,<br>parmis le top 10 des tokens."
    )
    fig.update_layout(barmode="stack")
    fig.update_layout(template="plotly")
    fig.write_html(f"stackedbar_regions_{task_name}.html")
    return fig

In [44]:
import pandas as pd 
from tqdm import tqdm
import torch
from residualstream_utils import remove_spaces

In [62]:
df_path = "data/countries.csv"
tasks = ["ISO_Code", "Dialing_Code", "Continent", "Capital", "Population"]
task = "Dialing_Code"

res = compute_stackbar_region(model, tokenizer, df_path, task)

[195] Task : Dialing Code: 100%|██████████| 196/196 [00:26<00:00,  7.49it/s, value=Zimbabwe]                        


In [46]:
df = pd.read_csv("results/result_Capital_top10_p0.01.csv")

In [47]:
df[df["Layer"] == -1]

,Unnamed: 0,Layer,Region,Country_Name,Capital,Prediction,Details
106,106,-1,Oceania,Marshall Islands,Majuro,Rajoy,[Marshall Islands] :[Marshall Islands] [Mars...
164,164,-1,Asia,Sri Lanka,Sri Jayawardenepura Kotte,Colombo,[Sri Lanka] :[Sri Lanka] [Sri Lanka] S[Sri L...


In [57]:
plot_stackbar_region(res)